# Week 9: BoVW Pipeline — k-means, Distances, Classifier

**Topics:** Bag of Visual Words (BoVW), k-means visual vocabulary, distances
in BoVW space, PCA projection, SVM as a black-box classifier, confusion matrix.

This week we connect the two halves of an image pipeline:

1. **Features (week 7)** — SIFT gives a *variable* number of 128-dim points per image.
2. **Decisions (this week)** — convert that variable set into a *fixed-length* vector
   (BoVW), check the resulting representation, and feed it to a classifier.

The classifier itself is treated as a **black box** — its internals come in week 11–12.
Our focus is on whether the features themselves carry class information.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

%matplotlib inline


### Environment setup

Works **locally** and on **Google Colab**. We use STL-10 via `torchvision`
instead of static image files — the dataset binary (~2.5 GB) is downloaded
into a local cache the first time the notebook runs (a few minutes).


In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    DATA_DIR = "/content/stl10"
else:
    DATA_DIR = os.path.expanduser("~/.cache/stl10")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"STL-10 cache: {DATA_DIR}")


### Display helpers

In [ ]:
def show_image(img, title=None, scale=4):
    """Display a single image. Grayscale arrays auto-use gray colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_keypoints(img, keypoints, title=None, scale=5):
    """Draw SIFT-style keypoints (circle = scale, line = orientation)."""
    out = cv2.drawKeypoints(
        img, keypoints, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )
    fig, ax = plt.subplots(figsize=(scale, scale))
    ax.imshow(out)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_bovw_histogram(hist, title=None, scale=(8, 3)):
    """Plot a single BoVW histogram (length K = number of visual words)."""
    fig, ax = plt.subplots(figsize=scale)
    K = len(hist)
    ax.bar(np.arange(K), hist, width=1.0)
    ax.set_xlim([-0.5, K - 0.5])
    ax.set_xlabel("Visual word index")
    ax.set_ylabel("Frequency")
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_scatter_2d(points, labels, class_names, title=None, scale=6):
    """Scatter 2D points colored by class label, with a legend."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    for cls in range(len(class_names)):
        mask = labels == cls
        ax.scatter(points[mask, 0], points[mask, 1], s=20,
                   color=plt.cm.tab10(cls), label=class_names[cls],
                   edgecolor="k", linewidth=0.4)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


def show_confusion_matrix(cm, class_names, title=None, scale=5):
    """Display a confusion matrix as an annotated heatmap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046)
    n = len(class_names)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=10)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


---

## 0. STL-10 Dataset

**STL-10** is CIFAR-10's larger sibling: 10 classes of 96×96 RGB images.
The larger resolution matters — at 32×32 SIFT struggles to find keypoints
(no room for the DoG octaves), but at 96×96 SIFT works comfortably.

We use a 5-class subset (vehicles + animals) so the confusion matrix stays
readable.


In [ ]:
from torchvision.datasets import STL10

train_ds = STL10(root=DATA_DIR, split="train", download=True)
test_ds = STL10(root=DATA_DIR, split="test", download=True)

print(f"len(train_ds) = {len(train_ds)}")
print(f"len(test_ds)  = {len(test_ds)}")
print(f"classes       = {train_ds.classes}")


Inspect one sample to see what `train_ds[i]` returns:

In [ ]:
img0, lab0 = train_ds[0]
print(f"img0 type   = {type(img0).__name__}")
print(f"img0 size   = {img0.size}     # PIL gives (W, H)")
print(f"img0 mode   = {img0.mode}")
print(f"lab0        = {lab0}  -> {train_ds.classes[lab0]}")


Pick 5 classes (`airplane`, `bird`, `car`, `cat`, `ship`) and remap their
labels to `0..4` so downstream confusion matrices and class names line up.

In [ ]:
SELECTED = [0, 1, 2, 3, 8]
class_names = [train_ds.classes[c] for c in SELECTED]
print(f"class_names = {class_names}")

PER_CLASS_TRAIN = 100
PER_CLASS_TEST = 30


def select_subset(ds, n_per_class):
    """Filter ds to the SELECTED classes and remap labels to 0..len(SELECTED)-1."""
    new_label = {old: i for i, old in enumerate(SELECTED)}
    items = []
    counts = {c: 0 for c in SELECTED}
    for i in range(len(ds)):
        img, label = ds[i]
        if label not in new_label:
            continue
        if counts[label] >= n_per_class:
            continue
        gray = np.array(img.convert("L"))   # uint8 (96, 96)
        items.append((gray, new_label[label]))
        counts[label] += 1
        if all(v >= n_per_class for v in counts.values()):
            break
    return items


train_items = select_subset(train_ds, PER_CLASS_TRAIN)
test_items = select_subset(test_ds, PER_CLASS_TEST)

print(f"train_items = {len(train_items)}")
print(f"test_items  = {len(test_items)}")


One sample image per class:

In [ ]:
for cls in range(len(class_names)):
    for gray, lab in train_items:
        if lab == cls:
            show_image(gray, title=f"{class_names[cls]} (label={cls})")
            break


---

## 1. SIFT Descriptors (review from week 7)

SIFT detects keypoints and emits one **128-dim descriptor** per keypoint.
A descriptor is a point in 128-dim space — exactly the picture from week 7.

The number of keypoints **varies per image** (different content → different
counts). That variability is the core problem this week solves.


In [ ]:
sift = cv2.SIFT_create()

gray, _ = train_items[0]
kp, des = sift.detectAndCompute(gray, None)

print(f"len(kp)   = {len(kp)}")
print(f"des.shape = {des.shape}")
print(f"des.dtype = {des.dtype}")


In [ ]:
show_keypoints(gray, kp, title=f"{len(kp)} SIFT keypoints on first train image")


Run SIFT on **every** training image. Each keypoint has a `.response`
score — keep only the top `N_DESC_CAP` per image so a few cluttered images
don't dominate the visual vocabulary later.

In [ ]:
N_DESC_CAP = 100  # keep top-N keypoints per image (by response)

train_desc = []
y_train = []
for gray, lab in train_items:
    kp, des = sift.detectAndCompute(gray, None)
    if des is None or len(des) == 0:
        train_desc.append(np.zeros((0, 128), dtype=np.float32))
        y_train.append(lab)
        continue
    if len(des) > N_DESC_CAP:
        responses = np.array([k.response for k in kp])
        top = np.argpartition(-responses, N_DESC_CAP)[:N_DESC_CAP]
        des = des[top]
    train_desc.append(des.astype(np.float32))
    y_train.append(lab)
y_train = np.array(y_train)

counts = np.array([len(d) for d in train_desc])
print(f"len(train_desc) = {len(train_desc)}")
print(f"y_train.shape   = {y_train.shape}")
print(f"descriptors/image  min = {counts.min()},  median = {int(np.median(counts))},  max = {counts.max()}")


Stack everything into one big `(N, 128)` array — that is what k-means consumes.

In [ ]:
all_descriptors = np.vstack([d for d in train_desc if len(d) > 0])
print(f"all_descriptors.shape = {all_descriptors.shape}")


---

## 2. k-means → Visual Vocabulary

k-means partitions the 128-dim descriptor space into **K** Voronoi cells.
Each centroid is one **visual word**:

$$ \{c_1, \dots, c_K\} = \arg\min_{c} \sum_i \min_k \|x_i - c_k\|^2 $$

For speed we use `MiniBatchKMeans` (the same algorithm, fit on small batches).


In [ ]:
import time
from sklearn.cluster import MiniBatchKMeans

K_VOCAB = 200

t0 = time.time()
kmeans = MiniBatchKMeans(n_clusters=K_VOCAB, batch_size=1024, n_init=3, random_state=0)
kmeans.fit(all_descriptors)
print(f"fit time: {time.time() - t0:.1f}s")


Inspect what `kmeans` carries forward:

In [ ]:
print(f"type(kmeans)               = {type(kmeans).__name__}")
print(f"cluster_centers_.shape     = {kmeans.cluster_centers_.shape}   # (K, 128)")
print(f"cluster_centers_.dtype     = {kmeans.cluster_centers_.dtype}")

# .predict turns descriptors into word indices in [0, K)
demo_words = kmeans.predict(train_desc[0])
print(f"demo_words.shape           = {demo_words.shape}")
print(f"demo_words[:10]            = {demo_words[:10]}")


---

## 3. BoVW Encoding — Variable Count → Fixed Length

For each image:

1. Compute its descriptors.
2. Assign each descriptor to its nearest visual word (`kmeans.predict`).
3. Count how often each word appears → length-K histogram.
4. L1-normalize so two images with different keypoint counts are comparable.

The output is an `(N_images, K)` matrix — a **fixed-length** representation
that any classifier can consume.


In [ ]:
def encode_bovw(per_image_desc, kmeans, K):
    H = np.zeros((len(per_image_desc), K), dtype=np.float32)
    for i in range(len(per_image_desc)):
        des = per_image_desc[i]
        if len(des) == 0:
            continue
        words = kmeans.predict(des)
        H[i] = np.bincount(words, minlength=K)
    sums = H.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1.0
    return H / sums


X_train = encode_bovw(train_desc, kmeans, K_VOCAB)
print(f"X_train.shape = {X_train.shape}")
print(f"X_train[0, :10] = {X_train[0, :10]}")
print(f"row sum check = {X_train[0].sum():.4f}")


Show the BoVW histogram for one image of each class:

In [ ]:
for cls in range(len(class_names)):
    idx = np.where(y_train == cls)[0][0]
    show_bovw_histogram(X_train[idx], title=f"BoVW histogram — {class_names[cls]} (sample)")


---

## 4. Distances in BoVW Space

A BoVW vector is just a list of K numbers — by itself, not very illuminating.
What is informative is the **distance** between two such vectors. If our
features are good, then:

- Two images of the **same** class should land **close** in BoVW space.
- Two images of **different** classes should land **far** apart.

To check this concretely, we compute all pairwise distances and find the
triplet `(i, j, k)` where image `i` and image `j` (same class) have the
*smallest* distance while image `i` and image `k` (different class) have the
*largest* — i.e. the most dramatic same-vs-different contrast.


In [ ]:
from sklearn.metrics import pairwise_distances

D = pairwise_distances(X_train, metric="euclidean")
print(f"D.shape = {D.shape}   # all-pairs L2 distances between BoVW vectors")
print(f"D[0, :5] = {D[0, :5]}")


Search every same-class pair `(i, j)` and find the most distant
different-class image `k`. Restrict to images with at least 30 descriptors so
we don't pick noisy outliers.

In [ ]:
best_ratio = 0.0
best = None
for i in range(len(X_train)):
    if counts[i] < 30:
        continue
    same = (y_train == y_train[i]) & (counts >= 30)
    same[i] = False
    diff = (y_train != y_train[i]) & (counts >= 30)
    if not same.any() or not diff.any():
        continue
    same_idx = np.where(same)[0]
    diff_idx = np.where(diff)[0]
    j_cand = same_idx[np.argmin(D[i, same_idx])]
    k_cand = diff_idx[np.argmax(D[i, diff_idx])]
    if D[i, j_cand] < 1e-6:
        continue
    ratio = D[i, k_cand] / D[i, j_cand]
    if ratio > best_ratio:
        best_ratio = ratio
        best = (i, j_cand, k_cand)

i, j, k = best
print(f"i = {i}  ({class_names[y_train[i]]})")
print(f"j = {j}  ({class_names[y_train[j]]})  same class as i")
print(f"k = {k}  ({class_names[y_train[k]]})  different class")
print(f"D(i, j) same-class    = {D[i, j]:.4f}")
print(f"D(i, k) different     = {D[i, k]:.4f}")
print(f"ratio                 = {best_ratio:.2f}x")


The three images and their BoVW vectors. The same-class pair shares
its visual-word distribution; the different-class image lights up an entirely
different set of words.

In [ ]:
show_image(train_items[i][0], title=f"i = {i}  ({class_names[y_train[i]]})")
show_image(train_items[j][0], title=f"j = {j}  ({class_names[y_train[j]]})  — same class")
show_image(train_items[k][0], title=f"k = {k}  ({class_names[y_train[k]]})  — different class")


In [ ]:
print(f"X_train[i] (first 20 of {K_VOCAB} dims) =\n{X_train[i, :20]}")
print(f"X_train[j] (first 20 of {K_VOCAB} dims) =\n{X_train[j, :20]}")
print(f"X_train[k] (first 20 of {K_VOCAB} dims) =\n{X_train[k, :20]}")


In [ ]:
show_bovw_histogram(X_train[i], title=f"BoVW(i)  — {class_names[y_train[i]]}")
show_bovw_histogram(X_train[j], title=f"BoVW(j)  — {class_names[y_train[j]]}  (same class as i)")
show_bovw_histogram(X_train[k], title=f"BoVW(k)  — {class_names[y_train[k]]}  (different class)")


---

## 5. PCA — Project to 2D for Visualization

K = 200 dimensions is impossible to plot directly. **PCA** finds the 2 axes
that preserve the most variance and projects every BoVW vector onto them.

Why PCA and not t-SNE? PCA is **parametric** — it returns a linear projection
matrix, so we can apply it to new points (and unproject back). t-SNE produces
only a fixed embedding of the points it saw.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0).fit(X_train)

print(f"components_.shape          = {pca.components_.shape}     # (2, K)")
print(f"explained_variance_ratio_  = {pca.explained_variance_ratio_}")
print(f"total variance kept        = {pca.explained_variance_ratio_.sum():.3f}")


In [ ]:
X_train_2d = pca.transform(X_train)
print(f"X_train_2d.shape = {X_train_2d.shape}")

show_scatter_2d(X_train_2d, y_train, class_names,
                title="BoVW features (training set) projected to PCA 2D")


Notice how much the classes **overlap**. Two PCA dimensions only retain
a small fraction of the total variance, so a lot of class-relevant structure
hides in the other 198 dimensions. This is why the next section's classifier
operates on the **full** 200-dim BoVW, not on the 2D projection.

---

## 6. Classifier as a Black Box → Confusion Matrix

> **Black box for now.** We use an SVM (Support Vector Machine) as a generic
> classifier — `.fit(X, y)` learns, `.predict(X)` predicts. The internals
> (margin maximization, kernels) come in week 11–12.

We push the test set through the same pipeline (SIFT → BoVW with the
**same** `kmeans` vocabulary) and ask the SVM to predict each test image's
class. The confusion matrix shows which classes get confused with which.


In [ ]:
test_desc = []
y_test = []
for gray, lab in test_items:
    kp, des = sift.detectAndCompute(gray, None)
    if des is None or len(des) == 0:
        test_desc.append(np.zeros((0, 128), dtype=np.float32))
        y_test.append(lab)
        continue
    if len(des) > N_DESC_CAP:
        responses = np.array([k.response for k in kp])
        top = np.argpartition(-responses, N_DESC_CAP)[:N_DESC_CAP]
        des = des[top]
    test_desc.append(des.astype(np.float32))
    y_test.append(lab)
y_test = np.array(y_test)

X_test = encode_bovw(test_desc, kmeans, K_VOCAB)
print(f"X_test.shape = {X_test.shape}, y_test.shape = {y_test.shape}")


In [ ]:
from sklearn.svm import SVC

clf = SVC(kernel="rbf", gamma="scale", C=10).fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)
print(f"train accuracy = {train_acc:.3f}")
print(f"test accuracy  = {test_acc:.3f}")


`cm[i, j]` = number of test images of true class `i` predicted as class
`j`. Diagonal entries are correct predictions; off-diagonal entries are
the failures, broken down by which class the model fell into.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(f"cm.shape = {cm.shape}, cm.dtype = {cm.dtype}")
print(f"cm =\n{cm}")


In [ ]:
show_confusion_matrix(cm, class_names, title=f"Confusion matrix (test acc = {test_acc:.2f})")


Per-class **precision**, **recall**, and **F1**:

- Precision = of all images predicted as class `c`, what fraction are truly `c`?
- Recall = of all images truly `c`, what fraction did we predict as `c`?
- F1 = harmonic mean of the two.

In [ ]:
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))


---

## 7. Exercises


**Exercise 7.1 — Find the best K (vocabulary size).**

The vocabulary size `K` is a hyperparameter. Re-run the full BoVW + SVM
pipeline at three values of `K`: **50, 200, 500** — and pick the best one
by **test accuracy**.

For each `K` you need to:

1. Re-fit `MiniBatchKMeans(n_clusters=K, ...)` on `all_descriptors`.
2. Re-encode train and test images via `encode_bovw(...)`.
3. Re-fit `SVC(kernel="rbf", gamma="scale", C=10)` on the new BoVW features.
4. Score on the test set.

A simple `for K in [50, 200, 500]:` loop is enough.

**Discussion:** Which K wins? What happens to the **PCA scatter** as K changes
— do the classes look more or less clustered?


In [ ]:
# YOUR CODE HERE
# Produce three lists/values:
#   K_list   = [50, 200, 500]
#   acc_list = test accuracy at each K  (float)
#   pca_list = PCA-2D-projected X_train at each K  (np.ndarray of shape (N, 2))


# --- Output display ---
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(K_list, acc_list, marker="o")
ax.set_xlabel("K (vocabulary size)")
ax.set_ylabel("Test accuracy")
ax.set_title("BoVW test accuracy vs. K")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for K_val, a in zip(K_list, acc_list):
    print(f"K = {K_val:>3d}  ->  test acc = {a:.3f}")

for K_val, X2 in zip(K_list, pca_list):
    show_scatter_2d(X2, y_train, class_names, title=f"PCA 2D — K = {K_val}")


# K가 너무 작으면 / 너무 크면 어떤 trade-off가 발생하는가?
# Test accuracy 기준 best K는 무엇이고, PCA scatter에서 그 K가 시각적으로도
# class를 가장 잘 분리하는가?
# (You may write your answer in Korean.)
#


**Exercise 7.2 — Raw pixel baseline.**

Skip BoVW entirely. Take each 96×96 grayscale training image, **flatten** it
into a length-9216 vector, and feed those vectors directly to the same
`SVC(kernel="rbf", gamma="scale", C=10)`. Compare the resulting test
accuracy to the BoVW result from Section 6.

This shows what BoVW's invariances buy us: same classifier, same images,
with vs without the SIFT/k-means preprocessing.

*Hints:*
- The training images are in `train_items` as `(gray, label)` tuples.
- Use `np.stack` and `.reshape(N, -1)` to build `X_pix_train`.
- Apply the **same flattening** to `test_items` for `X_pix_test`.


In [ ]:
# YOUR CODE HERE
# Produce these variables:
#   X_pix_train: (N_train, 9216) float32 — flattened grayscale training images
#   X_pix_test:  (N_test,  9216) float32 — flattened grayscale test images
#   clf_pix:     trained SVC on raw pixels
#   pix_test_acc: float — test accuracy of clf_pix


# --- Output display ---
print(f"BoVW       test accuracy = {test_acc:.3f}")
print(f"Raw-pixel  test accuracy = {pix_test_acc:.3f}")


# BoVW는 왜 raw pixel보다 잘 (또는 비슷하게) 동작하는가?
# 어떤 invariance(이동, 회전, 스케일, 조명 등)를 BoVW가 얻었는가?
# (You may write your answer in Korean.)
#


---

## Summary

- **BoVW** turns a variable-count set of 128-dim SIFT descriptors into a
  fixed-length K-bin histogram via nearest visual word — a representation
  any classifier can consume.
- **k-means** builds the visual vocabulary by partitioning descriptor space
  into K Voronoi cells; each centroid is one visual word.
- **Distances in BoVW space** carry semantic structure: same-class images
  are typically closer than different-class images.
- **PCA** is the right projection for visualization — it is parametric, and
  reveals how separable (or not) the classes are at a glance.
- **Confusion matrix + precision / recall / F1** tell us *where* the
  classifier fails, not just *how often* — crucial for diagnosis.

**Next (week 10):** instead of designing features (SIFT) and clustering
(k-means) by hand, can a neural network *learn* a feature extractor end-to-end
from labeled images? That is the MLP / CNN story.
